<a href="https://colab.research.google.com/github/24f3003058/DL-GenAI/blob/main/Week2/CustomerChurnPrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader,TensorDataset
from sklearn.model_selection import train_test_split,StratifiedKFold
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.metrics import (classification_report,confusion_matrix,roc_auc_score,
                             roc_curve,precision_recall_curve,f1_score,precision_score,recall_score)
from sklearn.utils.class_weight import compute_class_weight

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


#set random seeds for reproducibility
np.random.seed(75)
torch.manual_seed(75)
if torch.cuda.is_available():
  torch.cuda.manual_seed(75)

#configure visualization
plt.style.use('default')
plt.rcParams['figure.figsize']=(12,8)
plt.rcParams['font.size']=12
sns.set_palette('husl')

#check device availability
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Environment setup")
print(f"Device : {device}")
print(f"Pytorch version : {torch.__version__}")
print(f"Pandas version : {pd.__version__}")
print(f"Numpy version : {np.__version__}")

if torch.cuda.is_available():
  print(f" GPU: {torch.cuda.get_device_name(0)}")
  print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

print("\n All libraries imported successfully!")
print("Ready to begin customer churn analysis...")

Environment setup
Device : cuda
Pytorch version : 2.11.0+cu130
Pandas version : 2.2.3
Numpy version : 2.1.3
 GPU: Tesla T4
GPU memory: 14.6 GB

 All libraries imported successfully!
Ready to begin customer churn analysis...


In [ ]:
!pip install gdown
!gdown https://drive.google.com/uc?id=1xM2bFllWdNVVge0awuTKXvY_grM78vXD

Downloading...
From: https://drive.google.com/uc?id=1xM2bFllWdNVVge0awuTKXvY_grM78vXD
To: /content/Telco-Customer-Churn.csv
100% 970k/970k [00:00<00:00, 133MB/s]


# Data Loading

In [ ]:
data=pd.read_csv('CustomerChurn.csv')

In [ ]:
# Try different potential locations for the data file
data_paths = [ 'data/Telco-Customer-Churn.csv',
              ' .. /data/Telco-Customer-Churn. csv',
               ' .. / .. /data/Telco-Customer-Churn.csv',
               'Telco-Customer-Churn.csv']

In [ ]:
df=None
for path in data_paths:
  try:
    df=pd.read_csv(path)
    print(f"data loaded successfully from :{path}")
    break
  except FileNotFoundError:
    continue

if df is None:
  print("Dataset not found. Please ensure 'Telco-Customer-Churn.csv' is in the data directory ")
  print("You can download it from : https://www.kaggle.com/datasets/blastchar/telco-customer-churn")

  np.random_seed(42)
  n_samples=7043
  df = pd.DataFrame({
        'customerID': [f'ID_{i:04d}' for i in range(n_samples)],
        'gender': np.random.choice(['Male', 'Female'], n_samples),
        'SeniorCitizen': np.random.choice([0, 1], n_samples, p=[0.84, 0.16]),
        'Partner': np.random.choice(['Yes', 'No'], n_samples, p=[0.52, 0.48]),
        'Dependents': np.random.choice(['Yes', 'No'], n_samples, p=[0.30, 0.70]),
        'tenure': np.random.randint(0, 73, n_samples),
        'PhoneService': np.random.choice(['Yes', 'No'], n_samples, p=[0.90, 0.10]),
        'MultipleLines': np.random.choice(['Yes', 'No', 'No phone service'], n_samples, p=[0.42, 0.48, 0.10]),
        'InternetService': np.random.choice(['DSL', 'Fiber optic', 'No'], n_samples, p=[0.34, 0.44, 0.22]),
        'OnlineSecurity': np.random.choice(['Yes', 'No', 'No internet service'], n_samples, p=[0.28, 0.50, 0.22]),
        'OnlineBackup': np.random.choice(['Yes', 'No', 'No internet service'], n_samples, p=[0.34, 0.44, 0.22]),
        'DeviceProtection': np.random.choice(['Yes', 'No', 'No internet service'], n_samples, p=[0.34, 0.44, 0.22]),
        'TechSupport': np.random.choice(['Yes', 'No', 'No internet service'], n_samples, p=[0.29, 0.49, 0.22]),
        'StreamingTV': np.random.choice(['Yes', 'No', 'No internet service'], n_samples, p=[0.38, 0.40, 0.22]),
        'StreamingMovies': np.random.choice(['Yes', 'No', 'No internet service'], n_samples, p=[0.38, 0.40, 0.22]),
        'Contract': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples, p=[0.55, 0.21, 0.24]),
        'PaperlessBilling': np.random.choice(['Yes', 'No'], n_samples, p=[0.59, 0.41]),
        'PaymentMethod': np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)'],
                                        n_samples, p=[0.34, 0.19, 0.22, 0.25]),
        'MonthlyCharges': np.random.uniform(18.25, 118.75, n_samples),
        'TotalCharges': np.random.uniform(18.8, 8684.8, n_samples),
        'Churn': np.random.choice(['Yes', 'No'], n_samples, p=[0.27, 0.73])
    })
print("✅ Sample data created for demonstration")
print("DATASET OVERVIEW")
print(f" Shape: {df.shape}")


data loaded successfully from :Telco-Customer-Churn.csv
✅ Sample data created for demonstration
DATASET OVERVIEW
 Shape: (7043, 21)


In [ ]:
# Display first few rows
print(f"\n👀 First 5 rows:")
print(df.head())


👀 First 5 rows:
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies    

In [ ]:
# Check data types and missing values
print(f"\n🔍 Data Types and Missing Values:")
info_df = pd.DataFrame({
    'Column': df.columns,
    'Data Type': df.dtypes,
    'Non-Null Count': df.count(),
    'Null Count': df.isnull().sum(),
    'Null %': (df.isnull().sum() / len(df) * 100).round(2),
    'Unique Values': df.nunique()
})
print(info_df.to_string(index=False))

# Check target variable distribution
print(f"\n🎯 Target Variable (Churn) Distribution:")
churn_counts = df['Churn'].value_counts()
churn_percentages = df['Churn'].value_counts(normalize=True) * 100

for category, count in churn_counts.items():
    percentage = churn_percentages[category]
    print(f"   {category}: {count:,} customers ({percentage:.1f}%)")

# Calculate class imbalance ratio
minority_class = churn_counts.min()
majority_class = churn_counts.max()
imbalance_ratio = majority_class / minority_class
print(f"   Class imbalance ratio: {imbalance_ratio:.1f}:1")

if imbalance_ratio > 3:
    print("   ⚠️  Significant class imbalance detected - will need special handling")
else:
    print("   ✅ Mild class imbalance - standard techniques should work")


# Basic statistics for numerical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'customerID' in numerical_cols:
    numerical_cols.remove('customerID')
print(f"\n📈 Numerical Columns Summary:")
print(df[numerical_cols].describe())

print(f"\n🏷️ Categorical Columns Summary:")
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
if 'customerID' in categorical_cols:
    categorical_cols.remove('customerID')

for col in categorical_cols[:5]:  # Show first 5 categorical columns
    unique_count = df[col].nunique()
    print(f"   {col}: {unique_count} unique values - {list(df[col].unique())}")

if len(categorical_cols) > 5:
    print(f"   ... and {len(categorical_cols) - 5} more categorical columns")

print(f"\n✅ Initial data exploration completed!")
print(f"🔍 Key observations:")
print(f"   • Dataset contains {df.shape[0]:,} customer records")
print(f"   • {len(categorical_cols)} categorical and {len(numerical_cols)} numerical features")
print(f"   • Target variable shows {imbalance_ratio:.1f}:1 class imbalance")
print(f"   • Data quality check: {df.isnull().sum().sum()} total missing values")



🔍 Data Types and Missing Values:
          Column Data Type  Non-Null Count  Null Count  Null %  Unique Values
      customerID    object            7043           0     0.0           7043
          gender    object            7043           0     0.0              2
   SeniorCitizen     int64            7043           0     0.0              2
         Partner    object            7043           0     0.0              2
      Dependents    object            7043           0     0.0              2
          tenure     int64            7043           0     0.0             73
    PhoneService    object            7043           0     0.0              2
   MultipleLines    object            7043           0     0.0              3
 InternetService    object            7043           0     0.0              3
  OnlineSecurity    object            7043           0     0.0              3
    OnlineBackup    object            7043           0     0.0              3
DeviceProtection    object    

# Data Cleaning and Quality Assesment

In [ ]:
print("Data Cleaning and Quality Assessment")
df_cleaned=df.copy()
original_shape=df_cleaned.shape
print(f"Starting with {original_shape[0]:,} rows and {original_shape[1]} columns")

print(f"\n Checking for duplicate customers")
duplicate_customers=df_cleaned['customerID'].duplicated().sum()
print(f" Duplicate customerIDs: {duplicate_customers}")

if duplicate_customers >0:
  print("Removing duplicate customer records..")
  df_cleaned=df_cleaned.drop_duplicates(subset=['customerID'])
  print(f"Removed {duplicate_customers} duplicate records")
print(f'\n Fixing TotalCharges data type..')
print(f" Current TotalCharges type: {df_cleaned['TotalCharges'].dtype}")

if df_cleaned['TotalCharges'].dtype =='object':
  non_numeric_total=df_cleaned['TotalCharges'].apply(lambda x: not str(x).replace('.','').repalce(' ',"").isdigit())
  non_numeric_count=non_numeric_total.sum()
  print(f"Non-numeric TotalCharges values :{non_numeric_count}")

  if non_numeric_count >0:
    print('Converting TotalCharge to numeric..')
    missing_values=df_cleaned.isnull().sum()
    missing_percent=(missing_values/len(df_cleaned)*100).round(2)
    missing_summary=pd.DataFrame({
        'Column':missing_values.index,
        'Missing '
    })

Data Cleaning and Quality Assessment
Starting with 7,043 rows and 21 columns

 Checking for duplicate customers
 Duplicate customerIDs: 0
